Elaboró:

ROJAS MARTINEZ JONATHAN FRANCISCO

# Algoritmos genéticos

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [20]:
# Valor en bots - Valor en decimal - Aptitud - probabilidad de seleccion 

In [21]:
# Función para generar un numero binario de longitud de n_bits
def num_bin_ale(n_bits):
    return [random.randint(0, 1) for _ in range(n_bits)]

In [22]:
# Función para convertir un numero binario a decimal
def decodificar_binario(bits, a, b):
    n = len(bits)
    # Convertir lista de bits a string y luego a entero decimal
    decimal = int("".join(str(bit) for bit in bits), 2)
    # Interpolar en el intervalo [a, b] (normalizar)
    return a + decimal * (b - a) / ((2**n) - 1)

In [23]:
# Funcion de aptitud
def fitness_function(x):
    return -x**3 + 60*x**2 + 15000

In [24]:
# Seleccion de los mas fuertes
def seleccion(valores, num_seleccionados):
    # ordenamos de mayor a menor la parte de aptitudes y obtenemos los indices
    indices_ordenados = valores[:, 2].argsort()[::-1]
    valores = valores[indices_ordenados]
    # seleccionamos los mejores individuos
    seleccionados = [valores[i] for i in range(num_seleccionados)]
    return seleccionados

In [25]:
# Cruza entre 2 individuos
def cruza(in1, in2, n_bits):
    # 1. Hacemos copias independientes para no alterar a los padres originales
    hijo1 = in1.copy()
    hijo2 = in2.copy()
    
    # Escogemos una cantidad de puntos de cruza aleatorios
    puntos_cruza = random.randint(1, n_bits - 1) # Sin el 0 porque queremos 1 o mas puntos de cruza
    
    # Las posiciones empezando de atrás para adelante
    for punto in range(1, puntos_cruza + 1):
        # Intercambiamos los bits aleatoriamente entre los dos que tengan
        accion = random.choice([0, 1])
        if accion == 0:
            # Modificamos solo a los hijos
            hijo1[-punto], hijo2[-punto] = hijo2[-punto], hijo1[-punto]
            
    # Retornamos las nuevas listas
    return hijo1, hijo2

In [26]:
# Ruleta de nuevo nacimiento 
# Tenemos n individuos cada uno con una probablidad de ser seleccionado 
def ruleta_funcrandom(valores, num_seleccionados):
    # Seleccionamos un individuo basado en las probabilidades de seleccion
    seleccionados = np.random.choice(valores[:, 0], p=valores[:, 3].astype(float), size=num_seleccionados)
    return seleccionados

In [27]:
def mutacion(individuo):
    posicion = random.randrange(len(individuo))
    individuo[posicion] = 1 - individuo[posicion]  # Cambia el bit (0 a 1 o 1 a 0)
    return individuo

In [28]:
def creacion_de_generacion(poblacion, n_bits):
    return [num_bin_ale(n_bits) for _ in range(poblacion)]

In [49]:
# Selección de los mejores:
def probabilidades_de_seleccion(aptitudes):
    total_aptitud = np.sum(aptitudes)
    probabilidades = np.array(aptitudes) / total_aptitud
    return probabilidades

In [51]:
# --- 1. PREPARACIÓN Y PARÁMETROS INICIALES ---
n_bits = 6
intervalo = [0, 63]
tamano_poblacion = 6
generaciones_totales = 3  # Puedes aumentar esto a 50 o 100 después
probabilidad_mutacion = 0.15 # 15% de probabilidad de que un hijo mute

# print("==================================================")
# print("       INICIANDO ALGORITMO GENÉTICO")
# print("==================================================\n")

In [52]:
# Inicializamos la primera generación (solo cadenas de bits)
poblacion_actual = creacion_de_generacion(tamano_poblacion, n_bits)

# Creando las generaciones
for gen in range(generaciones_totales):
    print(f"--- GENERACIÓN {gen + 1} ---")
    
    # Convertimos los bits en los datos necesarios para evaluar (decimal, aptitud, probabilidad)
    datos_poblacion = crear_datos_gen(poblacion_actual, intervalo)
    
    print("1. Evaluación:")
    for ind in datos_poblacion:
        # Imprimimos de forma limpia: Bits | Decimal | Aptitud | Probabilidad
        print(f"   Bits: {ind[0]} | Decimal: {ind[1]:.2f} | Aptitud: {ind[2]:.2f} | Probabilidad: {ind[3]:.4f}")

    # SELECCIÓN
    # Hacemos que el numero de padres sea la mitad de la generación
    num_padres = int(tamano_poblacion/2)
    # Usamos la función de ruleta para escoger quiénes se reproducen
    # La ruleta ya tiende a escoger a los de mayor probabilidad, escogemos la mitad
    padres_seleccionados = ruleta_funcrandom(datos_poblacion, num_padres)
    print("\n2. Selección (Padres elegidos para reproducirse):")
    for p in padres_seleccionados:
        print(f"   {p}")

    # CRUZA
    siguiente_generacion = []
    print("\n3. Cruza:")
    # Tomamos a los padres de 2 en 2
    for i in range(0, num_padres):
        padre1 = padres_seleccionados[i].copy()
        # Evitamos error de índice si la población es impar
        padre2 = padres_seleccionados[i+1].copy() if (i+1) < num_padres else padres_seleccionados[0].copy()
        
        hijo1, hijo2 = cruza(padre1, padre2, n_bits)
        siguiente_generacion.extend([hijo1, hijo2])
        print(f"   Cruzando {padre1} y {padre2} -> Hijos: {hijo1}, {hijo2}")

    # MUTACIÓN
    print("\n4. Mutación:")
    for i in range(len(siguiente_generacion)):
        # Solo mutamos si caemos dentro de la probabilidad
        if random.random() < probabilidad_mutacion:
            antes_mutar = siguiente_generacion[i].copy()
            siguiente_generacion[i] = mutacion(siguiente_generacion[i])
            print(f"   ¡Mutación! El individuo {i} cambió de {antes_mutar} a {siguiente_generacion[i]}")
    
    # REEMPLAZO
    # Los hijos se convierten en la población actual para el siguiente ciclo
    poblacion_actual = siguiente_generacion
    print(f"--------------------------------------------------\n")



--- GENERACIÓN 1 ---
1. Evaluación:
   Bits: [0, 1, 0, 0, 1, 1] | Decimal: 19.00 | Aptitud: 29801.00 | Probabilidad: 0.1513
   Bits: [0, 1, 0, 1, 0, 1] | Decimal: 21.00 | Aptitud: 32199.00 | Probabilidad: 0.1635
   Bits: [1, 1, 0, 0, 0, 0] | Decimal: 48.00 | Aptitud: 42648.00 | Probabilidad: 0.2165
   Bits: [0, 0, 0, 0, 0, 1] | Decimal: 1.00 | Aptitud: 15059.00 | Probabilidad: 0.0765
   Bits: [1, 0, 0, 0, 1, 0] | Decimal: 34.00 | Aptitud: 45056.00 | Probabilidad: 0.2288
   Bits: [0, 1, 0, 1, 0, 1] | Decimal: 21.00 | Aptitud: 32199.00 | Probabilidad: 0.1635

2. Selección (Padres elegidos para reproducirse):
   [0, 0, 0, 0, 0, 1]
   [1, 0, 0, 0, 1, 0]
   [0, 1, 0, 0, 1, 1]

3. Cruza:
   Cruzando [0, 0, 0, 0, 0, 1] y [1, 0, 0, 0, 1, 0] -> Hijos: [0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 0]
   Cruzando [1, 0, 0, 0, 1, 0] y [0, 1, 0, 0, 1, 1] -> Hijos: [1, 0, 0, 0, 1, 0], [0, 1, 0, 0, 1, 1]
   Cruzando [0, 1, 0, 0, 1, 1] y [0, 0, 0, 0, 0, 1] -> Hijos: [0, 1, 0, 0, 1, 1], [0, 0, 0, 0, 0, 1]

4. Mu

In [53]:
# --- 3. RESULTADO FINAL ---
print("================== RESULTADO FINAL ==================")
datos_finales = crear_datos_gen(poblacion_actual, intervalo)
# Buscamos al individuo con la aptitud más alta (índice 2 de tu arreglo)
mejor_individuo = max(datos_finales, key=lambda x: x[2])

print(f"El mejor individuo encontrado tras {generaciones_totales} generaciones es:")
print(f"Bits: {mejor_individuo[0]}")
print(f"Valor Decimal: {mejor_individuo[1]:.2f}")
print(f"Aptitud Alcanzada: {mejor_individuo[2]:.2f}")

================== RESULTADO FINAL ==================
El mejor individuo encontrado tras 3 generaciones es:
Bits: [1, 0, 0, 0, 1, 0]
Valor Decimal: 34.00
Aptitud Alcanzada: 45056.00


- En general sacar las graficas de todos los algoritmos 
- Cauchy - graficas
- investigar para un polinomio como el de la clase resolverlo analíticamente para el intervalo que propongamos.
- tablas para todos los algoritmos
- el del lineal sacar las gráficas para ver las zonas y lineas de intersección, areas y linea punteada la de aptitud
- recodico mejorar la legibilidad del código y poner las funciones de temperatura junto con las graficas de donde van los puntos.
- Documentación de todos los códigos y como funcionan (2 - 3 hojas)